In [ ]:
import numpy as np
import pandas as pd

from other_dim_graphs import (
    prepare_dfs_to_plot,
    run_population_graphs_from_config,
)

# Mapping

In [ ]:
# mapping
bins_income = [
    -np.inf,   # for safety (shouldn't happen but robust)
    2000,
    4000,
    6000,
    8000,
    10000,
    12000,
    14000,
    16000,
    np.inf
]

labels_income = [1, 2, 3, 4, 5, 6, 7, 8, 9]

canton_to_number = {
    "Zurich": 1,
    "Bern": 2,
    "Lucerne": 3,
    "Uri": 4,
    "Schwyz": 5,
    "Obwalden": 6,
    "Nidwalden": 7,
    "Glarus": 8,
    "Zug": 9,
    "Fribourg": 10,
    "Solothurn": 11,
    "Basel_Stadt": 12,
    "Basel_Landschaft": 13,
    "Schaffhausen": 14,
    "Appenzell_Ausserrhoden": 15,
    "Appenzell_Innerrhoden": 16,
    "St_Gallen": 17,
    "Graubunden": 18,
    "Aargau": 19,
    "Thurgau": 20,
    "Ticino": 21,
    "Vaud": 22,
    "Valais": 23,
    "Neuchatel": 24,
    "Geneva": 25,
    "Jura": 26
}

def active_events_at_time(df: pd.DataFrame, t: float) -> pd.DataFrame:
    """maps active event of each dimension at time t and keeps attributes"""
    df = df.copy()

    # --- compute end dates
    main_end = df["main_start_date"] + df["main_duration"]
    gap_end  = df["gap_start_date"] + df["gap_duration"]

    # --- activity indicators
    df["main"] = ((df["main_start_date"] <= t) & (t < main_end)).astype(int)
    df["gap"]  = ((df["gap_start_date"] <= t) & (t < gap_end)).astype(int)

    # --- keep only active rows
    df_active = df[(df["main"] == 1) | (df["gap"] == 1)].copy()

    # --- resolve conflicts: one event per (id, dim_name)
    df_active["priority"] = df_active["main"]  # main > gap

    df_active = (
        df_active
        .sort_values(["id", "dim_name", "priority"], ascending=[True, True, False])
        .drop_duplicates(subset=["id", "dim_name"], keep="first")
        .drop(columns="priority")
    )

    # --- IMPORTANT: keep ALL columns
    # Just reorder so main/gap are visible
    cols_front = ["id", "dim_name", "event_name", "main", "gap"]
    other_cols = [c for c in df_active.columns if c not in cols_front]

    result = df_active[cols_front + other_cols].reset_index(drop=True)

    return result

def add_employment_status(df_active: pd.DataFrame, t: float) -> pd.DataFrame:
    df = df_active.copy()

    # --------------------------------------------------
    # 1. Get age from Existence
    # --------------------------------------------------
    existence = df[df["dim_name"] == "Existence"][["id", "main_start_date"]].copy()
    existence["age"] = t - existence["main_start_date"]

    # --------------------------------------------------
    # 2. Get Birth end date
    # --------------------------------------------------
    birth = df[df["event_name"] == "Birth"][["id", "main_start_date", "duration"]].copy()
    birth["birth_end"] = birth["main_start_date"] + birth["duration"]
    birth = birth[["id", "birth_end"]]

    # --------------------------------------------------
    # 3. Merge into Employment rows only
    # --------------------------------------------------
    emp = df[df["dim_name"] == "Employment"].copy()

    emp = emp.merge(existence[["id", "age"]], on="id", how="left")
    emp = emp.merge(birth, on="id", how="left")

    # compute end date of employment event
    emp["event_end"] = emp["main_start_date"] + emp["duration"]

    # --------------------------------------------------
    # 4. Initialize
    # --------------------------------------------------
    emp["employment_status"] = np.nan

    # --------------------------------------------------
    # 5. RULES
    # --------------------------------------------------

    # --- CASE 1: main = 1
    mask_main = emp["main"] == 1

    # 1a: employed (not NoWork)
    emp.loc[mask_main & (emp["event_name"] != "NoWork"), "employment_status"] = 1

    # 1b: NoWork
    mask_nowork = mask_main & (emp["event_name"] == "NoWork")
    emp.loc[mask_nowork & (emp["age"] < 15), "employment_status"] = 4
    emp.loc[mask_nowork & (emp["age"] >= 15), "employment_status"] = 2

    # --- CASE 2: gap = 1
    mask_gap = emp["gap"] == 1

    # default unemployed
    emp.loc[mask_gap, "employment_status"] = 2

    # retirement condition
    tol = 1e-6 
    mask_retired = (
        mask_gap
        & (emp["age"] >= 64)
        & np.isclose(emp["event_end"], emp["birth_end"], atol=tol)
    )
    emp.loc[mask_retired, "employment_status"] = 5

    # kids override (safety)
    emp.loc[emp["age"] < 15, "employment_status"] = 4

    # --------------------------------------------------
    # 6. Put back into dataframe
    # --------------------------------------------------
    df = df.merge(
        emp[["id", "dim_name", "event_name", "employment_status"]],
        on=["id", "dim_name", "event_name"],
        how="left"
    )

    return df

def add_education_status(df_active: pd.DataFrame) -> pd.DataFrame:
    df = df_active.copy()

    # --- select education rows
    edu = df[df["dim_name"] == "Education"].copy()

    # --- initialize
    edu["education_status"] = 0  # default = 0, finished / not in education

    # --- NoEducation corresponds to age < 6
    mask_no_education = edu["event_name"] == "NoEducation"
    edu.loc[mask_no_education, "education_status"] = 2

    # --- main education event, excluding NoEducation -> 1
    mask_main = (edu["main"] == 1) & (edu["event_name"] != "NoEducation")
    edu.loc[mask_main, "education_status"] = 1

    # --- merge back
    df = df.merge(
        edu[["id", "dim_name", "event_name", "education_status"]],
        on=["id", "dim_name", "event_name"],
        how="left"
    )

    return df

def collapse_to_individual(df_active: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "id",
        "canton_residence",
        "urban",
        "canton_work",
        "Income",
        "hh_income",
        "employment_status",
        "education_status",
        "has_license",
        "car_available",
        "has_ga",
        "age",
        "gender"
        
    ]
    
    cols = [c for c in cols if c in df_active.columns]

    return (
        df_active[cols]
        .groupby("id", sort=False, as_index=False)
        .first()
    )


def mapping(df_time_indep, t):
    # map active event in each dimension
    df_active_events = active_events_at_time(df_time_indep, t)

    # map variables to compare with df
    df_active_events["canton_residence"] = df_active_events["PlaceResidence"].map(canton_to_number)
    df_active_events["canton_work"] = df_active_events["PlaceWork"].map(canton_to_number)
    df_active_events["Income"] = np.exp(df_active_events["LogIncome"].astype(np.float64))
    df_active_events["hh_income"] = pd.cut(
        df_active_events["Income"],
        bins=bins_income,
        labels=labels_income,
        right=True,   # intervals like (a, b]
    ).astype("Int64")

    df_active_events = add_employment_status(df_active_events, t)
    df_active_events = add_education_status(df_active_events)

    #driving license
    mask_dl = (df_active_events["event_name"] == "DL") & (df_active_events["main"] == 1)

    df_active_events["has_license"] = (
        mask_dl.groupby(df_active_events["id"]).transform("any").astype(int)
    )
    
    mask_res = df_active_events["dim_name"] == "Residence"

    df_active_events.loc[mask_res, "car_available"] = (
        df_active_events.loc[mask_res, "has_license"] *
        df_active_events.loc[mask_res, "Car_availability"].fillna(0)
    ).astype(int)

    df_active_events["has_ga"] = df_active_events["AG_availability"]
    df_active_events["urban"] = df_active_events["Urban"]

    # select existence rows (already unique per id after dedup)
    existence = df_active_events[df_active_events["dim_name"] == "Existence"].copy()

    # compute age
    existence["age"] = t - existence["main_start_date"]

    # gender = Sex
    existence["gender"] = existence["Sex"]

    # keep only needed columns
    existence = existence[["id", "age", "gender"]]

    # merge back to all rows
    df_active_events = df_active_events.merge(existence, on="id", how="left")

    df_final = collapse_to_individual(df_active_events)
    return(df_final)


# Configurations

In [ ]:
COMMON_PLOTS_TO_RUN = [
    "income_by_gender",
    "income_by_age_group",
    "employment_status_table",
    "employment_status_by_gender_table",
    "license_status_table",
    "license_status_by_gender_table",
    "license_status_by_age_gender_table",
    "canton_residence",
    "canton_residence_urban",
    "employment_by_age_group",
    "employment_by_canton",
    "student_by_canton",
    "number_of_jobs_by_lifespan",
]

DATASET_CONFIGS = [
    {
        "path": r"posterior_2010_2015_sample.csv",
        "df_name": "posterior_2010_2015",
        "df_name_for_title": "Posterior 2010 2015",
        "df_type": "time_independent",
        "df_years_to_project": [2010, 2015],
        "df_year_observed": None,
        "weight_column": None,
        "weight_column_name": "weight",
    },
    {
        "path": r"prior_2000indiv.csv",
        "df_name": "prior",
        "df_name_for_title": "Prior -",
        "df_type": "time_independent",
        "df_years_to_project": [2010, 2015],
        "df_year_observed": None,
        "weight_column": None,
        "weight_column_name": "weight",
    },
    {
        "path": r"2010_sample.csv",
        "df_name": "MTMC_2010",
        "df_name_for_title": "MTMC in",
        "df_type": "time_dependent",
        "df_years_to_project": None,
        "df_year_observed": 2010,
        "weight_column": None,
        "weight_column_name": "weight",
    },
    {
        "path": r"2015_sample.csv",
        "df_name": "MTMC_2015",
        "df_name_for_title": "MTMC in",
        "df_type": "time_dependent",
        "df_years_to_project": None,
        "df_year_observed": 2015,
        "weight_column": None,
        "weight_column_name": "weight",
    },
]


# Run to get the graphs

In [ ]:
OUTPUT_FOLDER = "posterior_other_dim_graphs_automated"

all_results = {}

for cfg in DATASET_CONFIGS:
    print(f"\nRunning: {cfg['df_name']}")

    df = pd.read_csv(cfg["path"])

    dfs_to_plot, dfs_to_plot_years = prepare_dfs_to_plot(
        df=df,
        df_type=cfg["df_type"],
        df_years_to_project=cfg["df_years_to_project"],
        df_year_observed=cfg["df_year_observed"],
        weight_column=cfg["weight_column"],
        weight_column_name=cfg["weight_column_name"],
        mapping=mapping if cfg["df_type"] == "time_independent" else None,
        normalize_weights=True,
    )

    plot_config = {
        "df_name": cfg["df_name"],
        "df_name_for_title": cfg["df_name_for_title"],
        "df_type": cfg["df_type"],
        "dfs_to_plot_years": dfs_to_plot_years,
        "output_folder": OUTPUT_FOLDER,
        "weight_column_name": cfg["weight_column_name"],
        "plots_to_run": COMMON_PLOTS_TO_RUN,
    }

    result = run_population_graphs_from_config(
        dfs_to_plot=dfs_to_plot,
        config=plot_config,
        df=df,
    )

    all_results[cfg["df_name"]] = result

print("\nDone. Outputs saved in:", OUTPUT_FOLDER)

# Additional graph for driving license

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def clean_label(label):
    """
    Small helper to clean labels like 'Prior upd dob 2010 -'.
    """
    label = str(label).strip()
    label = label.rstrip("-").strip()
    return label


def merge_driving_license_tables(
    dataset_configs,
    input_dir,
    output_path=None,
):
    """
    Merge all driving-license-by-age-gender CSV tables into one long dataframe.

    Assumes each file is named:
        dl_status_by_age_gender_{config['df_name']}.csv

    Parameters
    ----------
    dataset_configs : list of dict
        Your DATASET_CONFIGS list.

    input_dir : str
        Folder where the dl_status_by_age_gender_*.csv files are stored.

    output_path : str or None
        If provided, saves the merged dataframe as a CSV.

    Returns
    -------
    df_all : pandas.DataFrame
        Long dataframe containing all datasets.
    """

    dfs = []

    for config in dataset_configs:
        df_name = config["df_name"]
        filename = f"dl_status_by_age_gender_{df_name}.csv"
        file_path = os.path.join(input_dir, filename)

        if not os.path.exists(file_path):
            print(f"Warning: file not found, skipped: {file_path}")
            continue

        df = pd.read_csv(file_path)

        # Add metadata identifying the dataset
        df["dataset_name"] = df_name
        df["dataset_label"] = clean_label(config["df_name_for_title"])
        df["df_type"] = config["df_type"]

        # Optional: keep only the years that correspond to the config
        if config["df_type"] == "time_independent":
            if config["df_years_to_project"] is not None:
                df = df[df["year"].isin(config["df_years_to_project"])]

        elif config["df_type"] == "time_dependent":
            if config["df_year_observed"] is not None:
                df = df[df["year"] == config["df_year_observed"]]

        dfs.append(df)

    if len(dfs) == 0:
        raise ValueError("No driving-license tables were found.")

    df_all = pd.concat(dfs, ignore_index=True)

    if output_path is not None:
        df_all.to_csv(output_path, index=False)

    return df_all

In [ ]:
INPUT_DIR = r"C:/Users/baud/Desktop/EPFL research/synthetic_population/population_outputs_from_scitas/new_posterior/other_dim_graphs_automated/posterior_other_dim_graphs_automated"

df_dl_all = merge_driving_license_tables(
    dataset_configs=DATASET_CONFIGS,
    input_dir=INPUT_DIR,
    output_path=os.path.join(INPUT_DIR, "dl_status_by_age_gender_all_datasets.csv"),
)



In [ ]:
def age_group_lower_bound(age_group):
    """
    Extract the lower bound from an age group such as '[25,35['.
    """
    return int(re.findall(r"\d+", str(age_group))[0])


def plot_driving_license_by_age_gender(
    df,
    value_col="weighted_percentage_license",
    year_col="year",
    age_col="age_group",
    gender_col="gender",
    dataset_col="dataset_label",
    gender_labels=None,
    output_dir=None,
):
    """
    One plot per year.
    x-axis: age group
    y-axis: driving-license ownership
    color: dataset
    line style: gender
    """

    if gender_labels is None:
        # CHANGE THIS if your coding is different
        gender_labels = {
            0.0: "Men",
            1.0: "Women",
        }

    df = df.copy()

    # Sort age groups properly
    age_order = (
        df[[age_col]]
        .drop_duplicates()
        .assign(age_lower=lambda d: d[age_col].apply(age_group_lower_bound))
        .sort_values("age_lower")[age_col]
        .tolist()
    )

    df[age_col] = pd.Categorical(
        df[age_col],
        categories=age_order,
        ordered=True,
    )

    df = df.sort_values([year_col, dataset_col, gender_col, age_col])

    datasets = df[dataset_col].drop_duplicates().tolist()
    years = sorted(df[year_col].drop_duplicates().tolist())

    # One color per dataset
    default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    color_by_dataset = {
        dataset: default_colors[i % len(default_colors)]
        for i, dataset in enumerate(datasets)
    }

    linestyle_by_gender = {
        "Men": "-",
        "Women": "--",
    }

    for year in years:
        df_year = df[df[year_col] == year]

        # Wider figure, but part of the width is reserved for the legend
        fig, ax = plt.subplots(figsize=(12, 6.5))

        for dataset in datasets:
            for gender_value, gender_label in gender_labels.items():

                sub = df_year[
                    (df_year[dataset_col] == dataset)
                    & (df_year[gender_col] == gender_value)
                ]

                if sub.empty:
                    continue

                ax.plot(
                    sub[age_col],
                    sub[value_col],
                    marker="o",
                    markersize=5,
                    linewidth=2.2,
                    color=color_by_dataset[dataset],
                    linestyle=linestyle_by_gender[gender_label],
                )

        ax.set_title(
            f"Driving-license ownership by age group and gender, {year}",
            fontsize=16,
            pad=12,
        )
        ax.set_xlabel("Age group", fontsize=14)
        ax.set_ylabel("Share with driving license (%)", fontsize=14)

        if value_col == "weighted_percentage_license":
            ax.set_ylim(0, 100)
        else:
            ax.set_ylim(0, 1)

        ax.grid(True, axis="y", alpha=0.3)

        ax.tick_params(axis="x", labelsize=12, rotation=35)
        ax.tick_params(axis="y", labelsize=12)

        # Reserve space on the right for legends.
        # This is the key change: the legend is inside the figure, not outside it.
        fig.subplots_adjust(
            left=0.09,
            right=0.66,
            bottom=0.20,
            top=0.88,
        )

        # Legend 1: datasets/colors
        dataset_handles = [
            Line2D(
                [0],
                [0],
                color=color_by_dataset[dataset],
                linewidth=2.5,
                label=dataset,
            )
            for dataset in datasets
        ]

        fig.legend(
            handles=dataset_handles,
            title="Dataset",
            title_fontsize=13,
            fontsize=11.5,
            loc="upper left",
            bbox_to_anchor=(0.70, 0.84),
            frameon=True,
        )

        # Legend 2: gender/line style
        gender_handles = [
            Line2D(
                [0],
                [0],
                color="black",
                linestyle=linestyle_by_gender[label],
                linewidth=2.5,
                label=label,
            )
            for label in gender_labels.values()
        ]

        fig.legend(
            handles=gender_handles,
            title="Gender",
            title_fontsize=13,
            fontsize=12,
            loc="upper left",
            bbox_to_anchor=(0.70, 0.30),
            frameon=True,
        )

        if output_dir is not None:
            os.makedirs(output_dir, exist_ok=True)

            fig.savefig(
                os.path.join(output_dir, f"driving_license_age_gender_{year}.pdf"),
                format="pdf",
                dpi=300,
            )

        plt.show()

In [ ]:
plot_driving_license_by_age_gender(
    df=df_dl_all,
    value_col="weighted_percentage_license",
    output_dir= r"C:/Users/baud/Desktop/EPFL research/synthetic_population/population_outputs_from_scitas/new_posterior/other_dim_graphs_automated/posterior_other_dim_graphs_automated",
)